# Grokking QA Overlay - WORKING VERSION

Uses the paper's **exact parameters** from their run_main_experiments.sh

In [ ]:
# Setup
!git clone https://github.com/LucasPrietoAl/grokking-at-the-edge-of-numerical-stability.git grokking_repo
%cd grokking_repo

In [ ]:
# Install deps
!pip install -q torch pandas matplotlib

In [ ]:
# Run baseline with PAPER'S EXACT PARAMS (this actually groks)
!python grokking_experiments.py \
  --lr 0.01 \
  --num_epochs 80000 \
  --log_frequency 5000 \
  --device cuda:0 \
  --train_fraction 0.4 \
  --loss_function stablemax \
  --cross_entropy_dtype float64 \
  --beta2 0.999

In [ ]:
# Check the results
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Find metrics file
metrics_files = list(Path('loggs').rglob('metrics.csv'))
if not metrics_files:
    print("ERROR: No metrics.csv found")
else:
    metrics_file = sorted(metrics_files, key=lambda p: p.stat().st_mtime)[-1]
    print(f"Reading: {metrics_file}")
    
    df = pd.read_csv(metrics_file)
    
    # Extract test accuracy
    test_acc = df[(df['input_type'] == 'test') & (df['metric_name'] == 'accuracy')]
    
    if len(test_acc) > 0:
        plt.figure(figsize=(10, 4))
        plt.plot(test_acc['epoch'], test_acc['value'])
        plt.xlabel('Epoch')
        plt.ylabel('Test Accuracy')
        plt.title('Does it grok?')
        plt.grid(True)
        plt.show()
        
        final_acc = test_acc['value'].iloc[-1]
        print(f"\nFinal test accuracy: {final_acc:.3f}")
        
        if final_acc > 0.9:
            print("✓ GROKKING OCCURRED!")
        else:
            print("✗ No grokking - try running longer or different seed")
    else:
        print("No test accuracy data found")